<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m05-lda-ica/lab/lab5_lda_ica.ipynb)

# Lab 5 — LDA and ICA

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import warnings
import numpy as np
import pandas as pd
import scipy.linalg
import scipy.signal
from scipy.stats import kurtosis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA, FastICA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.exceptions import ConvergenceWarning

**Course:** 040563107 Tools for Statistical Data Science · **Module m05** · **Time:** 3 hours · **Graded lab G3** (3⅓% of the course)

**After this lab you can:**
* compute Fisher's discriminant directions from the generalised eigenproblem, and explain why there are at most $K-1$ (objective m05-O1);
* fit LDA inside a pipeline and compare LDA and PCA features against baselines (O5);
* show how much an LDA fitted before the split inflates accuracy (O5, failure analysis);
* recover known sources from linear mixtures with FastICA and compare with PCA (O2, O3);
* demonstrate that ICA cannot identify Gaussian sources (O4, failure analysis).

**Two steps:** in the lab session we **run through** the ideas together. Then, as homework, you **do this lab on your own**
from a fresh copy and upload the `.ipynb` to the G3 assignment in Microsoft Teams. **That homework is what is marked.**

**How it is marked** ([lab rubric](https://github.com/pasdptt/public_teaching_tools4DS/blob/main/course/rubrics.md#lab-rubric-graded-labs)):
your notebook is re-run from top to bottom in a fresh session and checked with hidden tests (correctness 3 points,
reproducibility 1 point); staff mark your written answers (failure analysis 2, interpretation 2).

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes. Do not edit given cells or self-check cells.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before submitting, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

## Part 1 — Fisher's discriminant directions (about 35 minutes)

For $K$ classes with means $\bar x_k$, overall mean $\bar x$ and $n_k$ observations each, define the **within-class** and
**between-class** scatter matrices

$$S_W = \sum_k \sum_{i \in k} (x_i - \bar x_k)(x_i - \bar x_k)^\top, \qquad S_B = \sum_k n_k (\bar x_k - \bar x)(\bar x_k - \bar x)^\top .$$

Fisher's criterion $J(w) = \dfrac{w^\top S_B w}{w^\top S_W w}$ is maximised by the solutions of the **generalised eigenproblem**
$S_B w = \lambda S_W w$ with the largest $\lambda$. Because $S_B$ is a sum of $K$ rank-one matrices whose deviations sum to zero
(weighted by $n_k$), its rank is at most $K - 1$: there are at most $K - 1$ useful directions.

`scipy.linalg.eigh(S_B, S_W)` solves $S_B w = \lambda S_W w$ and returns eigenvalues in **ascending** order.

In [ ]:
Xw, yw = tsds.data.load("wine")
Zw = StandardScaler().fit_transform(Xw)
y_wine = yw.to_numpy()
print(Zw.shape, "classes:", np.unique(y_wine))

In [ ]:
# Task T1
# (a) fisher_directions(X, y): return (eigvals, W), the K-1 largest generalised eigenvalues of S_B w = lambda S_W w in
#     DESCENDING order and the matching eigenvectors as the columns of W (shape p x (K-1)).
# (b) fisher_vals, fisher_W = fisher_directions(Zw, y_wine)
def fisher_directions(X, y):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print("generalised eigenvalues:", np.round(fisher_vals, 4), "| W shape:", fisher_W.shape)

In [ ]:
# Self-check for task T1: run it; it should print OK.
assert fisher_W.shape == (13, 2), "Wine has 3 classes, so K - 1 = 2 directions"
assert fisher_vals[0] >= fisher_vals[1] > 0, "eigenvalues in descending order"
ours = Zw @ fisher_W
sk = LinearDiscriminantAnalysis(n_components=2).fit(Zw, y_wine).transform(Zw)
for j in range(2):
    r = abs(np.corrcoef(ours[:, j], sk[:, j])[0, 1])
    assert r > 0.999, f"direction {j + 1} should match scikit-learn's LDA up to sign and scale (|corr| = {r:.4f})"
print("OK")

**Q1.** **Interpretation.** Why does Wine give exactly two discriminant directions, however many variables it has? What do your two
eigenvalues say about how well each direction separates the cultivars, relative to the spread within cultivars?

*Your answer:* 

## Part 2 — Two features: PCA or LDA? (about 25 minutes)

Both reduce Wine's 13 standardised variables to **two** features; PCA ignores the labels, LDA uses them. Each is fitted
**inside** the pipeline, compared under the same repeated cross-validation, with two baselines.

In [ ]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
clf = LogisticRegression(max_iter=5000)

In [ ]:
# Task T2
# scores: dict name -> the 25 cross-validated accuracies on (Xw, yw) with cv=rcv for
#   "majority": DummyClassifier(strategy="most_frequent")
#   "all13":    make_pipeline(StandardScaler(), clf)
#   "pca2":     make_pipeline(StandardScaler(), PCA(2, random_state=SEED), clf)
#   "lda2":     make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(n_components=2), clf)
# YOUR CODE HERE
raise NotImplementedError()
pd.DataFrame({name: [s.mean(), s.std()] for name, s in scores.items()}, index=["mean", "sd"]).T.round(3)

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert set(scores) == {"majority", "all13", "pca2", "lda2"} and all(len(s) == 25 for s in scores.values())
assert abs(scores["majority"].mean() - 0.399) < 0.01
assert scores["lda2"].mean() > scores["pca2"].mean(), "two supervised directions should beat two unsupervised ones here"
print("OK")

**Q2.** **Interpretation.** Compare `pca2`, `lda2` and `all13` using the means and standard deviations. Why can LDA's two features do better
than PCA's two here, and what does LDA need that PCA does not?

*Your answer:* 

## Part 3 — LDA fitted before the split (about 20 minutes)

The course's old feature-extraction notebook fitted LDA on **all** the data before splitting (audit issue E1). How much does that
matter? Pure noise: $n = 100$, $p = 200$, three random classes, so honest accuracy is about $1/3$.

In [ ]:
gen = np.random.default_rng(SEED)
Xn = gen.normal(size=(100, 200))
yn = gen.integers(0, 3, size=100)

In [ ]:
# Task T3
# Using rcv and clf:
#   lda_before: LinearDiscriminantAnalysis(n_components=2) fitted on ALL of (Xn, yn), then CV of clf on the 2 projected features
#   lda_inside: CV of make_pipeline(LinearDiscriminantAnalysis(n_components=2), clf) on (Xn, yn)
# YOUR CODE HERE
raise NotImplementedError()
print(f"LDA before the split {lda_before:.3f} | inside the pipeline {lda_inside:.3f} | chance about 0.333")

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert lda_before > lda_inside + 0.3, "LDA uses y: fitted on all rows it should look far better than it is"
assert abs(lda_inside - 1 / 3) < 0.1, "inside the pipeline, noise should score near chance"
print("OK")

## Part 4 — Blind source separation: ICA vs PCA (about 40 minutes)

Three **known** source signals, a sine, a square wave and a sawtooth, are mixed by a fixed matrix $A$: we observe only
$X = S A^\top$ (three "microphones"). ICA assumes the sources are **independent and non-Gaussian** and looks for an unmixing that
makes the outputs as non-Gaussian as possible; PCA only decorrelates.

Recovered components come back in **any order, with any sign and any scale**. So we score a recovery by matching: for each true
source, the largest absolute correlation with any recovered component, averaged over the sources (1 = perfect).

In [ ]:
t = np.linspace(0, 8, 2000)
S_true = np.c_[np.sin(2 * t), np.sign(np.sin(3 * t)), scipy.signal.sawtooth(2 * np.pi * t)]
A_mix = np.array([[1.0, 1.0, 1.0], [0.5, 2.0, 1.0], [1.5, 1.0, 2.0]])
X_mix = S_true @ A_mix.T
print("excess kurtosis of the sources:", kurtosis(S_true, axis=0).round(3), "(a Gaussian has 0)")

In [ ]:
# Task T4
# (a) recovery_score(S_true, S_est): for each column of S_true, the maximum absolute Pearson correlation with any column of
#     S_est; return the mean over the columns of S_true.
# (b) ica_rec: recovery_score of FastICA(n_components=3, whiten="unit-variance", random_state=SEED, max_iter=1000)
#     fitted to X_mix (use .fit_transform); pca_rec: the same for PCA(3, random_state=SEED).
def recovery_score(S_true, S_est):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print(f"recovery: ICA {ica_rec:.3f} | PCA {pca_rec:.3f}")

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert np.isclose(recovery_score(S_true, S_true), 1.0)
assert np.isclose(recovery_score(S_true, -3 * S_true[:, ::-1]), 1.0), "order, sign and scale must not matter"
assert ica_rec > 0.99, "ICA should recover these non-Gaussian sources almost exactly"
assert pca_rec < ica_rec - 0.1, "PCA only decorrelates; it should not recover the sources"
print("OK")

**Q3.** **Interpretation.** Why does ICA recover the sources and PCA not? Name the three things ICA can never determine about a source, and
say why the scoring had to ignore them.

*Your answer:* 

## Part 5 — Failure analysis: Gaussian sources (about 25 minutes)

Replace the sources with **independent Gaussian** noise and mix them with the same $A$. Then run FastICA five times with different
`random_state` values, on both the non-Gaussian and the Gaussian mixtures, and compare (a) the recovery of the true sources and
(b) the agreement between runs. FastICA may warn that it did not converge on the Gaussian data; that warning is part of the answer.

In [ ]:
S_gauss = np.random.default_rng(SEED + 1).normal(size=(2000, 3))
X_gauss = S_gauss @ A_mix.T
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def run_ica(X, seed):
    return FastICA(n_components=3, whiten="unit-variance", random_state=seed, max_iter=1000).fit_transform(X)

In [ ]:
# Task T5
# For seeds 0, 1, 2, 3, 4:
#   rec_nongauss: list of recovery_score(S_true, run_ica(X_mix, seed))
#   rec_gauss:    list of recovery_score(S_gauss, run_ica(X_gauss, seed))
#   agree_gauss:  list of recovery_score(run_ica(X_gauss, 0), run_ica(X_gauss, seed)) for seeds 1, 2, 3, 4
#                 (how well each run reproduces run 0)
# YOUR CODE HERE
raise NotImplementedError()
print("non-Gaussian recovery:", np.round(rec_nongauss, 3))
print("Gaussian recovery:    ", np.round(rec_gauss, 3))
print("Gaussian agreement with run 0:", np.round(agree_gauss, 3))

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert len(rec_nongauss) == 5 and len(rec_gauss) == 5 and len(agree_gauss) == 4
assert min(rec_nongauss) > 0.99, "non-Gaussian sources: every run should recover them"
assert max(rec_gauss) < 0.9, "Gaussian sources: ICA should not recover them"
assert min(agree_gauss) < 0.95, "Gaussian sources: different runs should disagree"
print("OK")

**Q4.** **Failure analysis.** Explain both failures in this lab: (a) why ICA cannot recover Gaussian sources, using your recovery and
agreement numbers; (b) why LDA fitted before the split gave the accuracy it did in Part 3. For each, say how you would detect the
problem in a real analysis where the truth is unknown.

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *how scipy.linalg.eigh solves a generalised eigenproblem* | *used eigh(S_B, S_W); sorted the output myself* | *T1 self-check against scikit-learn* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Aeberhard, S. & Forina, M. (1992). Wine [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5PC7J (CC BY 4.0).